# 1. Dal modello alla chain, dalla chain all'agente

Questo notebook costruisce tre oggetti reali dell'ecosistema LangChain:

1. un `ChatOpenAI` invocato direttamente;
2. una chain LCEL composta con prompt e parser;
3. un agente LangChain che decide autonomamente quando chiamare un tool.

Tutte le chiamate usano il modello indicato in `.env`. Il notebook non importa codice dal progetto.

## Configurazione autocontenuta

Cerchiamo `.env` nella directory corrente e nei suoi genitori. Questo permette di avviare Jupyter sia dalla radice sia dalla cartella `notebooks`.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI


def find_env(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for directory in (current, *current.parents):
        candidate = directory / '.env'
        if candidate.is_file():
            return candidate
    raise FileNotFoundError('File .env non trovato nella directory corrente o nei genitori.')


ENV_FILE = find_env()
load_dotenv(ENV_FILE, override=False)
if not os.getenv('OPENAI_API_KEY'):
    raise RuntimeError(f'OPENAI_API_KEY non configurata in {ENV_FILE}')

MODEL_NAME = os.getenv('OPENAI_MODEL') or 'gpt-5.4-mini'
model = ChatOpenAI(
    model=MODEL_NAME,
    reasoning_effort='low',
    use_responses_api=True,
    store=False,
    max_retries=2,
)
print('Modello:', MODEL_NAME)

## 1. Invocazione diretta

`ChatOpenAI` implementa l'interfaccia `BaseChatModel`. Riceve messaggi tipizzati e restituisce un `AIMessage`. In questa forma non esistono ancora prompt riutilizzabili, tool o loop.

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

direct_response = model.invoke([
    SystemMessage(content='Sei un docente di sistemi agentici. Rispondi in italiano.'),
    HumanMessage(content='Distingui modello, chain e agente in tre frasi.'),
])
print(direct_response.text)

## 2. Una chain LCEL

LangChain Expression Language usa l'operatore `|` per comporre componenti `Runnable`. Il prompt produce messaggi, il modello produce un `AIMessage`, il parser estrae una stringa.

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ('system', 'Sei un tutor LangChain. Fornisci una spiegazione concreta e un esempio.'),
    ('human', 'Spiega {topic} a uno studente che conosce Python.'),
])
chain = prompt | model | StrOutputParser()

chain_response = chain.invoke({'topic': 'il pattern ReAct'})
print(chain_response)

## 3. Un agente con tool calling

La funzione Python diventa un tool grazie a `@tool`: tipi e docstring formano lo schema che il modello vede. `create_agent` costruisce sopra LangGraph il loop modello → tool → osservazione → modello.

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool


@tool
def rectangle_area(width: float, height: float) -> float:
    '''Calcola l'area esatta di un rettangolo date larghezza e altezza.'''
    if width <= 0 or height <= 0:
        raise ValueError('Le dimensioni devono essere positive.')
    return width * height


agent = create_agent(
    model=model,
    tools=[rectangle_area],
    system_prompt=(
        'Sei un assistente di calcolo. Usa il tool per ogni area richiesta; '
        'non fare il calcolo mentalmente.'
    ),
)
agent_result = agent.invoke({
    'messages': [
        {'role': 'user', 'content': 'Un rettangolo misura 12,5 m per 8 m. Qual è l\'area?'}
    ]
})
print(agent_result['messages'][-1].text)

## Ispezionare la traccia ReAct

La risposta finale nasconde il loop. I messaggi mostrano invece la richiesta strutturata, il risultato del tool e la conclusione.

In [ ]:
for index, message in enumerate(agent_result['messages']):
    name = type(message).__name__
    tool_calls = getattr(message, 'tool_calls', None)
    print(f'{index}: {name}')
    if tool_calls:
        print('   tool_calls =', tool_calls)
    elif name == 'ToolMessage':
        print('   osservazione =', message.content)
    else:
        print('   contenuto =', str(message.content)[:180])

assert any(getattr(message, 'tool_calls', None) for message in agent_result['messages'])
assert any(type(message).__name__ == 'ToolMessage' for message in agent_result['messages'])

## Esperimenti

- Aggiungi un secondo tool per il perimetro e chiedi entrambi i valori.
- Rendi ambigua la docstring e osserva come peggiora la scelta del tool.
- Usa `agent.stream(..., stream_mode='updates')` per vedere i nodi mentre vengono eseguiti.

Il passaggio decisivo è questo: una chain segue una topologia decisa dal programmatore; un agente sceglie dinamicamente la prossima azione entro i confini definiti dall'harness.